# Section 00 — OpenRouter semantic baseline research

This notebook investigates the research boundary:

normalized OCR evidence → LLM → semantic interpretation

It is not part of the production-shaped pipeline and does not define the final ActivityAnalyzer contract. Its purpose is to compare models, prompts, parameters, input representations, robustness, latency, schema reliability, token use, and cost before any technique is promoted.

OCR is noisy evidence. Icons may appear as characters, Vietnamese diacritics may be broken, toolbar/menu text may dominate, and an incorrect recognition can still have high OCR confidence. The semantic model must reason across multiple evidence items rather than trust one suspicious token.

## Section 01 — Central research configuration

Change experiment choices only in this cell. MODEL_ID intentionally has no default: select one exact OpenRouter model slug for a run. Do not use an auto-router for the main benchmark, and do not switch models between samples within a run.

INPUT_RESULTS_PATH accepts results.json or results.jsonl from the OCR pipeline. GROUND_TRUTH_PATH is optional and accepts CSV or JSONL.

In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean, median


SEMANTIC_NOTEBOOK_REVISION = "2026-10-02-openrouter-semantic-r1"
MODEL_ID = None
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 300

INPUT_RESULTS_PATH = None
GROUND_TRUTH_PATH = None
SMOKE_SAMPLE_ID = None
INPUT_FORMAT = "blocks_layout"  # combined_text | blocks_confidence | blocks_layout
MAX_OCR_BLOCKS = 200
MAX_INPUT_CHARACTERS = 12000

USE_STRUCTURED_OUTPUT = True
PROMPT_VERSION = "semantic_prompt_v1"
RUN_BATCH = False
BATCH_MAX_SAMPLES = None
CONTINUE_ON_ERROR = True
RESUME = True
SEMANTIC_RESUME_RUN_ID = None
SEMANTIC_WORK_ROOT = Path("/kaggle/working/screcaap/semantic_runs")

VALID_DOMAINS = {"work", "entertain", "idle", "other"}

if INPUT_FORMAT not in {"combined_text", "blocks_confidence", "blocks_layout"}:
    raise ValueError("INPUT_FORMAT is not supported.")
if MAX_OCR_BLOCKS < 1 or MAX_INPUT_CHARACTERS < 1000:
    raise ValueError("Input limits are too small for a useful semantic experiment.")
if BATCH_MAX_SAMPLES is not None and (
    not isinstance(BATCH_MAX_SAMPLES, int) or BATCH_MAX_SAMPLES < 1
):
    raise ValueError("BATCH_MAX_SAMPLES must be None or a positive integer.")
if SEMANTIC_RESUME_RUN_ID is not None and not RESUME:
    raise ValueError("SEMANTIC_RESUME_RUN_ID requires RESUME=True.")

print(f"Semantic notebook revision: {SEMANTIC_NOTEBOOK_REVISION}")
print(f"Prompt: {PROMPT_VERSION}")
print(f"Input format: {INPUT_FORMAT}")
print(f"Batch execution enabled: {RUN_BATCH}")

## Section 02 — OpenRouter setup

OpenRouter is accessed through the OpenAI-compatible Python client pointed at https://openrouter.ai/api/v1. The API key is loaded from OPENROUTER_API_KEY in the environment or from Kaggle Secrets with the same name. It is never printed or persisted.

The client dependency is installed only when the existing version is outside the declared compatible range. Official references: [OpenRouter quickstart](https://openrouter.ai/docs/quickstart) and [structured outputs](https://openrouter.ai/docs/guides/features/structured-outputs).

A missing secret raises an explanatory error. Set the secret in Kaggle or the environment, then rerun this section.

In [ ]:
from packaging.version import Version


OPENAI_CLIENT_MIN_VERSION = Version("1.0.0")
OPENAI_CLIENT_MAX_VERSION = Version("3.0.0")


def installed_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


openai_version = installed_version("openai")
openai_compatible = (
    openai_version != "not installed"
    and OPENAI_CLIENT_MIN_VERSION <= Version(openai_version) < OPENAI_CLIENT_MAX_VERSION
)
if not openai_compatible:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-cache-dir",
            "--upgrade-strategy",
            "only-if-needed",
            "openai>=1.0,<3.0",
        ]
    )

from openai import OpenAI


def load_openrouter_api_key():
    environment_key = os.environ.get("OPENROUTER_API_KEY")
    if environment_key:
        return environment_key

    try:
        from kaggle_secrets import UserSecretsClient

        kaggle_key = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    except Exception as exc:
        raise RuntimeError(
            "OPENROUTER_API_KEY was not found in the environment or Kaggle Secrets. "
            "Add the secret without placing it in notebook source or outputs."
        ) from exc

    if not kaggle_key:
        raise RuntimeError("Kaggle Secret OPENROUTER_API_KEY is empty.")
    return kaggle_key


OPENROUTER_API_KEY = load_openrouter_api_key()
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"OpenAI-compatible client: {installed_version('openai')}")
print("OpenRouter key loaded safely; value not displayed.")

## Section 03 — OCR result loader

The semantic experiment consumes normalized OCR artifacts rather than screenshots. It accepts materialized results.json or incremental results.jsonl from pipeline/notebooks/01_screen_activity_pipeline.ipynb.

For JSONL, later attempts replace earlier attempts by sample_id and only latest successful OCR rows become semantic inputs. If several candidate run outputs exist, set INPUT_RESULTS_PATH explicitly; the notebook does not guess which experiment to use.

In [ ]:
def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_number}: {exc}") from exc
            if not isinstance(row, dict):
                raise TypeError(f"Expected an object at {path}:{line_number}.")
            rows.append(row)
    return rows


def discover_result_candidates():
    roots = [Path("/kaggle/input"), Path("/kaggle/working/screcaap/runs")]
    json_candidates = []
    jsonl_candidates = []
    for root in roots:
        if not root.exists():
            continue
        json_candidates.extend(root.rglob("results.json"))
        jsonl_candidates.extend(root.rglob("results.jsonl"))

    materialized_parents = {path.parent.resolve() for path in json_candidates}
    fallback_jsonl = [
        path for path in jsonl_candidates if path.parent.resolve() not in materialized_parents
    ]
    return sorted(set(json_candidates + fallback_jsonl), key=lambda path: path.as_posix())


def load_ocr_rows(path):
    if path.suffix.lower() == ".jsonl":
        history = read_jsonl(path)
        latest_by_sample_id = {}
        for row in history:
            sample_id = row.get("sample_id")
            if isinstance(sample_id, str):
                latest_by_sample_id[sample_id] = row
        rows = list(latest_by_sample_id.values())
    elif path.suffix.lower() == ".json":
        with path.open("r", encoding="utf-8") as input_file:
            rows = json.load(input_file)
        if not isinstance(rows, list):
            raise TypeError("results.json must contain an array.")
    else:
        raise ValueError("INPUT_RESULTS_PATH must end in .json or .jsonl.")

    successful = [
        row
        for row in rows
        if row.get("status") == "ok" and isinstance(row.get("ocr"), dict)
    ]
    successful.sort(key=lambda row: int(row["sequence_index"]))
    sample_ids = [row["sample_id"] for row in successful]
    if len(sample_ids) != len(set(sample_ids)):
        raise ValueError("OCR input contains duplicate successful sample_id values.")
    if not successful:
        raise RuntimeError("No successful normalized OCR result was found.")
    return successful


if INPUT_RESULTS_PATH is None:
    result_candidates = discover_result_candidates()
    if len(result_candidates) != 1:
        candidate_text = "\n".join(f"  {path}" for path in result_candidates) or "  [none]"
        raise RuntimeError(
            "Set INPUT_RESULTS_PATH explicitly because automatic discovery did not find "
            f"exactly one canonical result file. Candidates:\n{candidate_text}"
        )
    input_results_path = result_candidates[0]
else:
    input_results_path = Path(INPUT_RESULTS_PATH)

if not input_results_path.exists() or not input_results_path.is_file():
    raise FileNotFoundError(f"OCR result file does not exist: {input_results_path}")

ocr_records = load_ocr_rows(input_results_path)
print(f"OCR input: {input_results_path}")
print(f"Successful OCR samples: {len(ocr_records)}")

## Section 04 — Optional ground truth

Ground truth joins to OCR results by sample_id. CSV and JSONL are supported. Rows may contain sample_id, image_file, general_context, application, domain, and notes. Domain must be one of work, entertain, idle, or other.

Ground truth is optional for a smoke test. When absent, accuracy and confusion metrics are skipped rather than fabricated.

In [ ]:
def load_ground_truth(path):
    if path is None:
        return {}
    path = Path(path)
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f"Ground-truth file does not exist: {path}")

    if path.suffix.lower() == ".csv":
        with path.open("r", encoding="utf-8-sig", newline="") as input_file:
            rows = list(csv.DictReader(input_file))
    elif path.suffix.lower() == ".jsonl":
        rows = read_jsonl(path)
    else:
        raise ValueError("GROUND_TRUTH_PATH must be a CSV or JSONL file.")

    by_sample_id = {}
    for row_number, row in enumerate(rows, start=1):
        sample_id = str(row.get("sample_id") or "").strip()
        domain = str(row.get("domain") or "").strip().lower()
        if not sample_id:
            raise ValueError(f"Ground-truth row {row_number} has no sample_id.")
        if sample_id in by_sample_id:
            raise ValueError(f"Duplicate ground-truth sample_id: {sample_id}")
        if domain not in VALID_DOMAINS:
            raise ValueError(
                f"Ground-truth row {row_number} has invalid domain {domain!r}."
            )
        by_sample_id[sample_id] = {
            "sample_id": sample_id,
            "image_file": row.get("image_file"),
            "general_context": row.get("general_context"),
            "application": row.get("application"),
            "domain": domain,
            "notes": row.get("notes"),
        }
    return by_sample_id


ground_truth_by_sample_id = load_ground_truth(GROUND_TRUTH_PATH)
print(f"Ground-truth rows: {len(ground_truth_by_sample_id)}")
if not ground_truth_by_sample_id:
    print("Accuracy metrics will be skipped until GROUND_TRUTH_PATH is configured.")

## Section 05 — Semantic input formatting

build_semantic_input keeps the input boundary explicit instead of dumping an arbitrary Python object into the prompt.

Selectable representations:

- combined_text: approximate OCR reading order only
- blocks_confidence: recognized text plus OCR scores
- blocks_layout: recognized text, scores, polygons, and screen dimensions

The original OCR artifact remains unchanged. Input caps only limit the request representation, and the payload reports when truncation occurred.

In [ ]:
def build_semantic_payload(ocr_record, mode=INPUT_FORMAT):
    ocr = ocr_record["ocr"]
    blocks = list(ocr.get("blocks") or [])
    selected_blocks = blocks[:MAX_OCR_BLOCKS]
    payload = {
        "sample_id": ocr_record["sample_id"],
        "representation": mode,
        "ocr_is_noisy_evidence": True,
        "truncated": len(blocks) > len(selected_blocks),
    }

    combined_text = str(ocr.get("combined_text") or "")
    if mode == "combined_text":
        payload["combined_text"] = combined_text
    elif mode == "blocks_confidence":
        payload["blocks"] = [
            {
                "text": str(block.get("text") or ""),
                "confidence": float(block.get("confidence", 0.0)),
            }
            for block in selected_blocks
        ]
    elif mode == "blocks_layout":
        image = ocr_record.get("image") or {}
        payload["screen"] = {
            "width": image.get("width", ocr.get("image_width")),
            "height": image.get("height", ocr.get("image_height")),
        }
        payload["blocks"] = [
            {
                "text": str(block.get("text") or ""),
                "confidence": float(block.get("confidence", 0.0)),
                "polygon": block.get("polygon"),
            }
            for block in selected_blocks
        ]
    else:
        raise ValueError(f"Unsupported semantic input mode: {mode}")

    rendered = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    while len(rendered) > MAX_INPUT_CHARACTERS and payload.get("blocks"):
        payload["blocks"].pop()
        payload["truncated"] = True
        rendered = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))

    if len(rendered) > MAX_INPUT_CHARACTERS and "combined_text" in payload:
        overflow = len(rendered) - MAX_INPUT_CHARACTERS
        keep = max(0, len(payload["combined_text"]) - overflow - 32)
        payload["combined_text"] = payload["combined_text"][:keep]
        payload["truncated"] = True
        rendered = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))

    if len(rendered) > MAX_INPUT_CHARACTERS:
        raise ValueError(
            "Semantic input still exceeds MAX_INPUT_CHARACTERS after safe truncation."
        )
    return payload


def build_semantic_input(ocr_record):
    return json.dumps(
        build_semantic_payload(ocr_record),
        ensure_ascii=False,
        indent=2,
    )


preview_input = build_semantic_input(ocr_records[0])
print(preview_input[:2000] + ("\n..." if len(preview_input) > 2000 else ""))

## Section 06 — Baseline prompt

semantic_prompt_v1 treats OCR as fallible evidence. It requires multiple supporting clues, permits other when categories do not fit, and forbids unsupported details. Prompt identity is recorded with every run so model comparisons can hold prompt/schema/parameters constant.

In [ ]:
SYSTEM_PROMPT = """You are classifying the visible laptop activity represented by noisy OCR evidence.

Rules:
- OCR characters may be wrong, icons may appear as text, Vietnamese may lose diacritics, and menus/toolbars may be irrelevant.
- OCR confidence is not a guarantee that recognition is correct.
- Do not classify from one isolated suspicious token; use multiple mutually supporting clues.
- Choose exactly one domain: work, entertain, idle, or other.
- Use other when the activity does not fit the known categories or evidence is insufficient.
- Do not invent applications, actions, topics, or intent unsupported by the evidence.
- Evidence entries must be short excerpts or observations grounded in the supplied OCR.
- Uncertainties must state important ambiguity or noise.
- Return only the requested JSON object.
""".strip()

print(f"Prompt version: {PROMPT_VERSION}")
print(SYSTEM_PROMPT)

## Section 07 — Strict semantic output schema

Compatible models can use OpenRouter structured outputs with strict JSON Schema. If the selected model/provider does not support response_format, set USE_STRUCTURED_OUTPUT=False deliberately; the same response is then parsed and validated locally. Malformed or schema-invalid output is reported as failure and is never silently accepted.

In [ ]:
SEMANTIC_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "domain": {"type": "string", "enum": sorted(VALID_DOMAINS)},
        "general_context": {"type": "string"},
        "application": {"type": "string"},
        "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
        "evidence": {"type": "array", "items": {"type": "string"}},
        "uncertainties": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "domain",
        "general_context",
        "application",
        "confidence",
        "evidence",
        "uncertainties",
    ],
    "additionalProperties": False,
}
STRUCTURED_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "screen_activity_semantics",
        "strict": True,
        "schema": SEMANTIC_JSON_SCHEMA,
    },
}


def parse_json_content(content):
    if not isinstance(content, str) or not content.strip():
        raise ValueError("The model returned no text content.")
    stripped = content.strip()
    fence = chr(96) * 3
    if stripped.startswith(fence) and stripped.endswith(fence):
        lines = stripped.splitlines()
        stripped = "\n".join(lines[1:-1]).strip()
    try:
        parsed = json.loads(stripped)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Model output is not valid JSON: {exc}") from exc
    if not isinstance(parsed, dict):
        raise TypeError("Semantic output must be a JSON object.")
    return parsed


def validate_semantic_output(value):
    expected_keys = set(SEMANTIC_JSON_SCHEMA["required"])
    actual_keys = set(value)
    if actual_keys != expected_keys:
        raise ValueError(
            f"Semantic output keys must equal {sorted(expected_keys)}; got {sorted(actual_keys)}."
        )
    if value["domain"] not in VALID_DOMAINS:
        raise ValueError(f"Invalid semantic domain: {value['domain']!r}")
    if not isinstance(value["general_context"], str) or not value["general_context"].strip():
        raise TypeError("general_context must be a non-empty string.")
    if not isinstance(value["application"], str) or not value["application"].strip():
        raise TypeError("application must be a non-empty string; use 'unknown' when needed.")
    confidence = value["confidence"]
    if isinstance(confidence, bool) or not isinstance(confidence, (int, float)):
        raise TypeError("confidence must be numeric.")
    if not 0.0 <= float(confidence) <= 1.0:
        raise ValueError("confidence must be between 0 and 1.")
    for field_name in ("evidence", "uncertainties"):
        if not isinstance(value[field_name], list) or not all(
            isinstance(item, str) for item in value[field_name]
        ):
            raise TypeError(f"{field_name} must be an array of strings.")
    return value


print(json.dumps(STRUCTURED_RESPONSE_FORMAT, ensure_ascii=False, indent=2))

## Section 08 — Single-sample OpenRouter smoke test

Select SMOKE_SAMPLE_ID or leave it as None to use the first OCR sample. The request prints the requested/reported model, validated semantic result, latency, token usage, and reported cost when OpenRouter supplies it. It never prints the key, request headers, or full provider payload.

MODEL_ID must be an explicit model slug. Structured-output support is model/provider-dependent and should be checked on the OpenRouter model page before enabling a benchmark.

In [ ]:
def response_as_dict(response):
    if hasattr(response, "model_dump"):
        return response.model_dump()
    if hasattr(response, "dict"):
        return response.dict()
    return {}


def request_semantics(client, ocr_record):
    semantic_input = build_semantic_input(ocr_record)
    request = {
        "model": MODEL_ID,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Interpret this OCR evidence:\n\n" + semantic_input,
            },
        ],
    }
    if USE_STRUCTURED_OUTPUT:
        request["response_format"] = STRUCTURED_RESPONSE_FORMAT
        request["extra_body"] = {"provider": {"require_parameters": True}}

    started = time.perf_counter()
    response = client.chat.completions.create(**request)
    latency_ms = (time.perf_counter() - started) * 1000.0

    if not response.choices:
        raise RuntimeError("OpenRouter returned no completion choice.")
    content = response.choices[0].message.content
    semantic_result = validate_semantic_output(parse_json_content(content))

    response_payload = response_as_dict(response)
    usage = response_payload.get("usage") or {}
    return {
        "semantic": semantic_result,
        "latency_ms": float(latency_ms),
        "model_requested": MODEL_ID,
        "model_reported": response_payload.get("model"),
        "prompt_version": PROMPT_VERSION,
        "input_format": INPUT_FORMAT,
        "input_tokens": usage.get("prompt_tokens"),
        "output_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
        "reported_cost": usage.get("cost"),
        "parse_schema_success": True,
    }


if not MODEL_ID or not isinstance(MODEL_ID, str):
    raise ValueError("Set MODEL_ID to one exact OpenRouter model slug in Section 01.")
if MODEL_ID in {"auto", "openrouter/auto"}:
    raise ValueError("Do not use an auto-router for the main semantic benchmark.")

if SMOKE_SAMPLE_ID is None:
    smoke_record = ocr_records[0]
else:
    matching_records = [
        record for record in ocr_records if record["sample_id"] == SMOKE_SAMPLE_ID
    ]
    if len(matching_records) != 1:
        raise ValueError(f"SMOKE_SAMPLE_ID was not found exactly once: {SMOKE_SAMPLE_ID}")
    smoke_record = matching_records[0]

smoke_semantic = request_semantics(openrouter_client, smoke_record)
print(f"Sample: {smoke_record['sample_id']}")
print(f"Requested model: {smoke_semantic['model_requested']}")
print(f"Reported model: {smoke_semantic['model_reported']}")
print(f"Latency: {smoke_semantic['latency_ms']:.1f} ms")
print(
    "Tokens: "
    f"in={smoke_semantic['input_tokens']}, "
    f"out={smoke_semantic['output_tokens']}, "
    f"total={smoke_semantic['total_tokens']}"
)
print(f"Reported cost: {smoke_semantic['reported_cost']}")
print(json.dumps(smoke_semantic["semantic"], ensure_ascii=False, indent=2))

## Section 09 — Batch evaluation scaffold

The batch boundary is sequential by design. Each row records sample identity, optional ground-truth domain, predicted semantics, latency, token usage, reported cost, and parse/schema success. Every row is appended and flushed before the next request.

RUN_BATCH remains False by default to prevent unintended API spend. Rate limits, retry/backoff policy, and safe concurrency must be understood before parallel requests are introduced.

In [ ]:
def append_jsonl_and_flush(file_handle, record):
    file_handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    file_handle.flush()
    os.fsync(file_handle.fileno())


def analyze_sample(client, ocr_record):
    sample_id = ocr_record["sample_id"]
    ground_truth = ground_truth_by_sample_id.get(sample_id)
    try:
        semantic_call = request_semantics(client, ocr_record)
        return {
            "sample_id": sample_id,
            "sequence_index": int(ocr_record["sequence_index"]),
            "status": "ok",
            "ground_truth_domain": ground_truth["domain"] if ground_truth else None,
            **semantic_call,
        }
    except Exception as exc:
        return {
            "sample_id": sample_id,
            "sequence_index": int(ocr_record["sequence_index"]),
            "status": "error",
            "ground_truth_domain": ground_truth["domain"] if ground_truth else None,
            "parse_schema_success": False,
            "error": {"type": type(exc).__name__, "message": str(exc)},
        }


def evaluate_dataset(client, records, predictions_jsonl_path, successful_sample_ids):
    total = len(records)
    with predictions_jsonl_path.open("a", encoding="utf-8") as predictions_file:
        for position, record in enumerate(records, start=1):
            sample_id = record["sample_id"]
            if sample_id in successful_sample_ids:
                print(f"[{position}/{total}] {sample_id} — SKIP — already successful")
                continue

            prediction = analyze_sample(client, record)
            append_jsonl_and_flush(predictions_file, prediction)
            if prediction["status"] == "ok":
                print(
                    f"[{position}/{total}] {sample_id} — OK — "
                    f"{prediction['semantic']['domain']} — "
                    f"{prediction['latency_ms']:.1f} ms"
                )
            else:
                error = prediction["error"]
                print(
                    f"[{position}/{total}] {sample_id} — ERROR — "
                    f"{error['type']}: {error['message']}"
                )
                if not CONTINUE_ON_ERROR:
                    raise RuntimeError(error["message"])


print("Batch scaffold defined. RUN_BATCH=False, so no batch API calls have been made.")

## Section 10 — Evaluation metrics

The summary computes schema success, latency percentiles, tokens, and reported cost. When ground truth exists it also computes domain accuracy, per-class accuracy, and a confusion matrix. Without ground truth those accuracy fields are None or empty rather than misleading estimates.

In [ ]:
def percentile(values, fraction):
    if not values:
        return None
    ordered = sorted(float(value) for value in values)
    if len(ordered) == 1:
        return ordered[0]
    position = (len(ordered) - 1) * fraction
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)


def summarize_predictions(rows):
    total = len(rows)
    successful = [row for row in rows if row.get("status") == "ok"]
    schema_successful = [row for row in rows if row.get("parse_schema_success") is True]
    latencies = [row["latency_ms"] for row in successful]

    evaluated = [
        row
        for row in successful
        if row.get("ground_truth_domain") in VALID_DOMAINS
    ]
    correct = [
        row
        for row in evaluated
        if row["semantic"]["domain"] == row["ground_truth_domain"]
    ]

    per_class = {}
    confusion = {
        actual: {predicted: 0 for predicted in sorted(VALID_DOMAINS)}
        for actual in sorted(VALID_DOMAINS)
    }
    for domain in sorted(VALID_DOMAINS):
        domain_rows = [row for row in evaluated if row["ground_truth_domain"] == domain]
        domain_correct = [
            row for row in domain_rows if row["semantic"]["domain"] == domain
        ]
        per_class[domain] = {
            "count": len(domain_rows),
            "accuracy": len(domain_correct) / len(domain_rows) if domain_rows else None,
        }
    for row in evaluated:
        confusion[row["ground_truth_domain"]][row["semantic"]["domain"]] += 1

    input_tokens = [row["input_tokens"] for row in successful if row.get("input_tokens") is not None]
    output_tokens = [row["output_tokens"] for row in successful if row.get("output_tokens") is not None]
    costs = [row["reported_cost"] for row in successful if isinstance(row.get("reported_cost"), (int, float))]

    return {
        "total": total,
        "successful": len(successful),
        "failed": total - len(successful),
        "json_schema_success_rate": len(schema_successful) / total if total else None,
        "domain_accuracy": len(correct) / len(evaluated) if evaluated else None,
        "evaluated_with_ground_truth": len(evaluated),
        "per_class_accuracy": per_class if evaluated else {},
        "confusion_matrix": confusion if evaluated else {},
        "mean_latency_ms": mean(latencies) if latencies else None,
        "p50_latency_ms": median(latencies) if latencies else None,
        "p95_latency_ms": percentile(latencies, 0.95),
        "total_input_tokens": sum(input_tokens) if input_tokens else None,
        "total_output_tokens": sum(output_tokens) if output_tokens else None,
        "total_reported_cost": sum(costs) if costs else None,
        "reported_cost_per_screenshot": sum(costs) / len(costs) if costs else None,
    }


print("Metric functions defined; accuracy will be skipped cleanly without ground truth.")

## Section 11 — Chained-collapse experiment design

A future controlled experiment should compare the same samples, model, prompt, schema, and parameters across three evidence sources:

A. Human/clean semantic description → LLM

B. OCR combined text → LLM

C. Structured OCR blocks with confidence/layout → LLM

A estimates the LLM's intrinsic semantic error under clean evidence. The gap from A to B/C estimates OCR propagation and representation effects. Clean evidence must be human-annotated or otherwise genuinely available; never fabricate clean OCR ground truth from the same noisy output being evaluated.

## Section 12 — Model comparison discipline

To compare models, change MODEL_ID only and create a separate run. Keep the same OCR dataset, ground truth, prompt version, schema, temperature, token limit, structured-output setting, and input representation.

Use an exact model slug. Do not use an auto-router for the main benchmark and do not randomly switch providers/models between samples. Record the reported model because provider routing may still affect reproducibility.

## Section 13 — Semantic experiment output

When RUN_BATCH=True, artifacts are written under /kaggle/working/screcaap/semantic_runs/<run_id>:

- run.json: reproducibility metadata, hashes, model, prompt, schema, and parameters
- predictions.jsonl: append-only, immediately flushed execution history
- predictions.json: latest row per sample in deterministic order
- errors.jsonl: unresolved latest failures
- summary.json: schema, accuracy, latency, token, and reported-cost metrics

Resume selection is explicit through SEMANTIC_RESUME_RUN_ID. Only successful rows are skipped; failures are retried. The API key and full provider payload are never persisted.

In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def atomic_write_json(path, value):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as output_file:
        json.dump(value, output_file, ensure_ascii=False, indent=2)
        output_file.flush()
        os.fsync(output_file.fileno())
    temporary_path.replace(path)


def atomic_write_jsonl(path, rows):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as output_file:
        for row in rows:
            output_file.write(json.dumps(row, ensure_ascii=False) + "\n")
        output_file.flush()
        os.fsync(output_file.fileno())
    temporary_path.replace(path)


def run_semantic_batch():
    if not RUN_BATCH:
        print("RUN_BATCH=False; no batch OpenRouter requests or output directory created.")
        return None
    if not MODEL_ID or MODEL_ID in {"auto", "openrouter/auto"}:
        raise ValueError("Set one explicit MODEL_ID before running the batch.")

    selected_records = (
        ocr_records[:BATCH_MAX_SAMPLES]
        if BATCH_MAX_SAMPLES is not None
        else ocr_records
    )
    input_fingerprint = hashlib.sha256(
        json.dumps(
            [
                {
                    "sample_id": row["sample_id"],
                    "sequence_index": row["sequence_index"],
                }
                for row in selected_records
            ],
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()
    prompt_hash = hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest()
    schema_hash = hashlib.sha256(
        json.dumps(SEMANTIC_JSON_SCHEMA, sort_keys=True).encode("utf-8")
    ).hexdigest()
    run_configuration = {
        "model_id": MODEL_ID,
        "temperature": TEMPERATURE,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "input_format": INPUT_FORMAT,
        "use_structured_output": USE_STRUCTURED_OUTPUT,
        "prompt_version": PROMPT_VERSION,
        "prompt_hash": prompt_hash,
        "schema_hash": schema_hash,
        "input_results_path": str(input_results_path),
        "input_fingerprint": input_fingerprint,
        "batch_max_samples": BATCH_MAX_SAMPLES,
    }

    SEMANTIC_WORK_ROOT.mkdir(parents=True, exist_ok=True)
    if SEMANTIC_RESUME_RUN_ID is not None:
        run_id = SEMANTIC_RESUME_RUN_ID
        run_directory = SEMANTIC_WORK_ROOT / run_id
        run_path = run_directory / "run.json"
        if not run_path.exists():
            raise FileNotFoundError(f"Cannot resume; run.json does not exist: {run_path}")
        with run_path.open("r", encoding="utf-8") as input_file:
            run_metadata = json.load(input_file)
        if run_metadata.get("configuration") != run_configuration:
            raise RuntimeError(
                "Resume configuration/input does not match run.json. Start a new run."
            )
        print(f"Resuming explicit semantic run: {run_id}")
    else:
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_semantic_v1")
        run_directory = SEMANTIC_WORK_ROOT / run_id
        run_directory.mkdir(parents=True, exist_ok=False)
        run_metadata = {
            "run_id": run_id,
            "started_at": utc_now_iso(),
            "semantic_notebook_revision": SEMANTIC_NOTEBOOK_REVISION,
            "python_version": sys.version.split()[0],
            "openai_client_version": installed_version("openai"),
            "input_count": len(selected_records),
            "configuration": run_configuration,
        }
        atomic_write_json(run_directory / "run.json", run_metadata)
        print(f"Created semantic run: {run_id}")

    predictions_jsonl_path = run_directory / "predictions.jsonl"
    predictions_jsonl_path.touch(exist_ok=True)
    history = read_jsonl(predictions_jsonl_path)
    successful_sample_ids = {
        row["sample_id"]
        for row in history
        if row.get("status") == "ok" and row.get("parse_schema_success") is True
    }

    evaluate_dataset(
        openrouter_client,
        selected_records,
        predictions_jsonl_path,
        successful_sample_ids,
    )

    all_history = read_jsonl(predictions_jsonl_path)
    latest_by_sample_id = {}
    for row in all_history:
        latest_by_sample_id[row["sample_id"]] = row
    selected_ids = {record["sample_id"] for record in selected_records}
    ordered_predictions = sorted(
        (
            row
            for sample_id, row in latest_by_sample_id.items()
            if sample_id in selected_ids
        ),
        key=lambda row: int(row["sequence_index"]),
    )
    errors = [
        {
            "sample_id": row["sample_id"],
            "sequence_index": row["sequence_index"],
            "error_type": (row.get("error") or {}).get("type"),
            "error_message": (row.get("error") or {}).get("message"),
        }
        for row in ordered_predictions
        if row.get("status") == "error"
    ]
    summary = summarize_predictions(ordered_predictions)
    summary["run_id"] = run_id
    summary["completed_at"] = utc_now_iso()

    atomic_write_json(run_directory / "predictions.json", ordered_predictions)
    atomic_write_jsonl(run_directory / "errors.jsonl", errors)
    atomic_write_json(run_directory / "summary.json", summary)
    run_metadata["last_completed_at"] = summary["completed_at"]
    atomic_write_json(run_directory / "run.json", run_metadata)

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"Semantic run directory: {run_directory}")
    return run_directory


semantic_run_directory = run_semantic_batch()